# C4 — Auto-Research Editing Loop (Colab, A100)

Runs the society-vs-blind-vs-static critic ablation inside a 10-step
auto-refinement editing loop, then renders the deliverable figures.

**Requirements:** an **A100 40GB** runtime (Runtime → Change runtime type → A100),
and an HF token whose account has **accepted the `black-forest-labs/FLUX.1-Kontext-dev`
license** (it is a gated model). On smaller GPUs pass `--cpu-offload`, or
`--editor instructpix2pix` (ungated) as a fallback.

Pipeline: `script/c4_refine.py` (run) → `scripts/analysis/c4_trajectory.py` +
`c4_qualitative.py` (figures in `results/figs/`).

## 1. Clone the repo + install the GPU stack

In [ ]:
# Clone (replace with your remote if different)
![ -d SyntheticAudience ] || git clone https://github.com/<your-org>/SyntheticAudience.git
%cd SyntheticAudience
!pip -q install -r requirements-colab.txt

## 2. Hugging Face login (FLUX.1-Kontext-dev is gated)

In [ ]:
from huggingface_hub import login
login()  # paste a token with access to FLUX.1-Kontext-dev + your private data repos

## 3. Fetch data (EVA + PARA)
`data/` is git-ignored and pulled from your private HF datasets via the repo's helper.
Optionally mount Drive first so `data/c4_edits` + `data/results` persist across sessions.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')  # optional persistence
!python scripts/fetch_from_hf.py eva para

## 4. GPU check

In [ ]:
import torch
p = torch.cuda.get_device_properties(0)
gb = p.total_memory / 1e9
print(f'{p.name}  {gb:.1f} GB')
if gb < 38:
    print('WARNING: <40GB — add --cpu-offload (slow) or use --editor instructpix2pix.')

## 5. Smoke test (2 images, 3 steps) — confirm the whole loop runs

In [ ]:
!python script/c4_refine.py --dataset eva --n-images 2 \
    --conditions static,blind,society --editor flux --steps 3 --candidates 2

## 6. Full run
~150 images per dataset × 4 conditions × 10 steps × K=3. Long (overnight-ish);
cached + resumable (`--resume`), shardable across GPUs (`--shard i/N`).

In [ ]:
!python script/c4_refine.py --dataset both --n-images 150 \
    --conditions static,blind,society,reward_only \
    --editor flux --steps 10 --candidates 3 --resume

## 7. Deliverables — trajectory curves, AUC, drift, diversity, qualitative grid

In [ ]:
%cd scripts/analysis
!python c4_trajectory.py
!python c4_qualitative.py
%cd ../..

In [ ]:
from IPython.display import Image as IImage, display
for f in ['c4_trajectory.png', 'c4_drift.png', 'c4_diversity.png', 'c4_qualitative.png']:
    display(IImage(filename=f'results/figs/{f}'))